In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

In [2]:
ss_ds = xr.open_dataset('sg195_Shilshole_Overnight_OCN_201__15May2024_timeseries.nc')
display(ss_ds)

#Align time with sg_data_point and apply offset (if needed)
adjusted_time = pd.to_datetime(ss_ds['time'].values) + pd.DateOffset(years=19, months=7, days=16)
ss_ds = ss_ds.assign_coords(time=('sg_data_point', adjusted_time))
ss_ds.time.values

<xarray.Dataset> Size: 305kB
Dimensions:                                   (gps_info: 24,
                                               sg_data_point: 1671,
                                               trajectory: 8, dive: 8)
Coordinates:
    ctd_time                                  (sg_data_point) datetime64[ns] 13kB ...
    ctd_depth                                 (sg_data_point) float32 7kB ...
    latitude                                  (sg_data_point) float32 7kB ...
    longitude                                 (sg_data_point) float32 7kB ...
  * trajectory                                (trajectory) int32 32B 1 2 ... 7 8
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/66)
    gps_info_dive_number                      (gps_info) int32 96B ...
    sg_data_point_dive_number                 (sg_data_point) int32 7kB ...
    log_gps_time                              (gps_info) datetime64[ns] 192B ...
    time                                      (sg_data_point) datetime64[ns] 13kB ...
    pressure                                  (sg_data_point) float32 7kB ...
    depth                                     (sg_data_point) float32 7kB ...
    ...                                        ...
    end_longitude                             (dive) float32 32B ...
    depth_avg_curr_east                       (dive) float32 32B ...
    depth_avg_curr_north                      (dive) float32 32B ...
    depth_avg_curr_qc                         (dive) |S1 8B ...
    latlong_qc                                (dive) |S1 8B ...
    glider                                    |S12 12B ...
Attributes: (12/48)
    project:                         Shilshole Overnight OCN 201, 15May2024
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 Shilshole Overnight OCN 201, 15May...
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-05-16T00:05:41Z
    uuid:                            4a3bb1ce-1319-11ef-a72f-f57473c7a252
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

array(['2024-05-15T16:27:37.250999936', '2024-05-15T16:27:42.239000064',
       '2024-05-15T16:27:47.571000064', ...,
       '2024-05-15T23:08:36.839999872', '2024-05-15T23:08:42.005000064',
       '2024-05-15T23:08:47.023999872'],
      shape=(1671,), dtype='datetime64[ns]')

In [3]:
ss_ds['PAR_470nm'] = ss_ds['eng_wlbb2fl_sig470nm']
ss_ds['particle_concentration_700nm'] = ss_ds['eng_wlbb2fl_sig700nm']
ss_ds['chlorophyll_695nm'] = ss_ds['eng_wlbb2fl_sig695nm']
ss_ds['dissolved_oxygen'] = ss_ds['aanderaa4330_dissolved_oxygen']
ss_ds['instrument_dissolved_oxygen'] = ss_ds['aanderaa4330_instrument_dissolved_oxygen']

# add metadata
ss_ds['PAR_470nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig470nm'
ss_ds['particle_concentration_700nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig700nm'
ss_ds['chlorophyll_695nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig695nm'
ss_ds['dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_dissolved_oxygen'
ss_ds['instrument_dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_instrument_dissolved_oxygen'


#Select the relevant variables
new_ss_ds = ss_ds[['time', 'depth', 'latitude', 'longitude','temperature', 'salinity', 'dissolved_oxygen', 'instrument_dissolved_oxygen', 'PAR_470nm', 'particle_concentration_700nm', 'chlorophyll_695nm']]

#Convert to DataFrame and save
new_ss_ds.to_dataframe().reset_index().to_csv('sg195_ss_timeseries_cleaned.csv', index=False)
new_ss_ds.to_netcdf('sg195_ss_timeseries_cleaned.nc')
display(ss_ds)


<xarray.Dataset> Size: 339kB
Dimensions:                                   (gps_info: 24,
                                               sg_data_point: 1671,
                                               trajectory: 8, dive: 8)
Coordinates:
    time                                      (sg_data_point) datetime64[ns] 13kB ...
    ctd_time                                  (sg_data_point) datetime64[ns] 13kB ...
    ctd_depth                                 (sg_data_point) float32 7kB 0.4...
    latitude                                  (sg_data_point) float32 7kB 47....
    longitude                                 (sg_data_point) float32 7kB -12...
  * trajectory                                (trajectory) int32 32B 1 2 ... 7 8
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/70)
    gps_info_dive_number                      (gps_info) int32 96B ...
    sg_data_point_dive_number                 (sg_data_point) int32 7kB ...
    log_gps_time                              (gps_info) datetime64[ns] 192B ...
    pressure                                  (sg_data_point) float32 7kB ...
    depth                                     (sg_data_point) float32 7kB 0.9...
    speed_gsm                                 (sg_data_point) float32 7kB ...
    ...                                        ...
    glider                                    |S12 12B ...
    PAR_470nm                                 (sg_data_point) float32 7kB 121...
    particle_concentration_700nm              (sg_data_point) float32 7kB 531...
    chlorophyll_695nm                         (sg_data_point) float32 7kB 470...
    dissolved_oxygen                          (sg_data_point) float32 7kB nan...
    instrument_dissolved_oxygen               (sg_data_point) float32 7kB nan...
Attributes: (12/48)
    project:                         Shilshole Overnight OCN 201, 15May2024
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 Shilshole Overnight OCN 201, 15May...
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-05-16T00:05:41Z
    uuid:                            4a3bb1ce-1319-11ef-a72f-f57473c7a252
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [4]:
ss_ds = xr.open_dataset('sg195_Shilshole_Overnight_OCN_201__15May2024_timeseries.nc')

#Apply time apply offset (if needed)
adjusted_start = pd.to_datetime(ss_ds['start_time'].values) + pd.DateOffset(years=19, months=7, days=16)
adjusted_end = pd.to_datetime(ss_ds['end_time'].values) + pd.DateOffset(years=19, months=7, days=16)
ss_ds = ss_ds.assign_coords(start_time=('dive', adjusted_start), end_time=('dive', adjusted_end))

ss_ds['U_DAC'] = ss_ds['depth_avg_curr_east']
ss_ds['V_DAC'] = ss_ds['depth_avg_curr_north']

# add metadata
ss_ds['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
ss_ds['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_ss_ds = ss_ds[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_ss_ds)

#Convert to DataFrame and save
new_ss_ds.to_dataframe().reset_index().to_csv('sg195_ss_DAC_timeseries_cleaned.csv', index=False)
new_ss_ds.to_netcdf('sg195_ss_DAC_timeseries_cleaned.nc')


<xarray.Dataset> Size: 320B
Dimensions:          (dive: 8)
Coordinates:
    start_time       (dive) datetime64[ns] 64B 2024-05-15T16:26:58 ... 2024-0...
    end_time         (dive) datetime64[ns] 64B 2024-05-15T16:56:05 ... 2024-0...
Dimensions without coordinates: dive
Data variables:
    U_DAC            (dive) float32 32B ...
    V_DAC            (dive) float32 32B ...
    start_latitude   (dive) float32 32B ...
    end_latitude     (dive) float32 32B ...
    start_longitude  (dive) float32 32B ...
    end_longitude    (dive) float32 32B ...
Attributes: (12/48)
    project:                         Shilshole Overnight OCN 201, 15May2024
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 Shilshole Overnight OCN 201, 15May...
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-05-16T00:05:41Z
    uuid:                            4a3bb1ce-1319-11ef-a72f-f57473c7a252
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6